# Paper Evaluation Notebook: Retrieval + Generation

This notebook runs the QBrain benchmark and exports paper-ready tables/figures.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif ROOT.name != "rag_lab":
    # fallback for direct execution from repo root
    ROOT = Path("d:/Qbrainpython/QBrain/rag_lab")

OUT_DIR = ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark"
RESULTS_TABLES = ROOT / "results" / "tables"
RESULTS_FIGS = ROOT / "results" / "figures"

RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_FIGS.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


## 1) Run benchmark
Adjust parameters if needed (`k`, `threshold`, `max_questions_per_srs`).


In [ ]:
cmd = [
    sys.executable,
    str(ROOT / "scripts" / "run_rag_benchmark.py"),
    "--k", "5",
    "--threshold", "0.72",
    "--max-questions-per-srs", "10",
]

print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Benchmark failed")


## 2) Load outputs


In [ ]:
overall = pd.read_csv(OUT_DIR / "overall_summary.csv")
retrieval_srs = pd.read_csv(OUT_DIR / "retrieval" / "retrieval_summary_by_srs.csv")
generation_srs = pd.read_csv(OUT_DIR / "generation" / "generation_summary_by_srs.csv")
e2e_srs = pd.read_csv(OUT_DIR / "e2e" / "e2e_summary_by_srs.csv")
failure_breakdown = pd.read_csv(OUT_DIR / "diagnostics" / "failure_breakdown.csv")
metrics_by_category = pd.read_csv(OUT_DIR / "diagnostics" / "metrics_by_category.csv")

display(overall)
display(retrieval_srs)
display(generation_srs)
display(e2e_srs)


## 3) Build paper tables


In [ ]:
overall_map = {row['metric']: row['value'] for _, row in overall.iterrows()}
paper_overall = pd.DataFrame([
    ["Retrieval Hit@5", overall_map.get("retrieval_hit@5", None)],
    ["Retrieval Precision@5", overall_map.get("retrieval_precision@5", None)],
    ["Retrieval Recall@5", overall_map.get("retrieval_recall@5", None)],
    ["Retrieval MRR", overall_map.get("retrieval_mrr", None)],
    ["Generation Accuracy (sim >= 0.72)", overall_map.get("generation_accuracy_sim>=0.72", None)],
    ["Generation Avg Similarity", overall_map.get("generation_avg_similarity", None)],
    ["End-to-End Success Rate", overall_map.get("e2e_success_rate", None)],
], columns=["Metric", "Value"])

paper_overall.to_csv(RESULTS_TABLES / "paper_overall_results.csv", index=False)

paper_per_srs = e2e_srs.merge(
    retrieval_srs[["srs_file", "hit_rate"]], on="srs_file", how="left"
).merge(
    generation_srs[["srs_file", "answer_accuracy"]], on="srs_file", how="left"
)

paper_per_srs = paper_per_srs[["srs_file", "hit_rate", "answer_accuracy", "e2e_success_rate"]]
paper_per_srs.columns = ["SRS File", "Retrieval Hit@5", "Generation Accuracy", "E2E Success"]
paper_per_srs.to_csv(RESULTS_TABLES / "paper_per_srs_results.csv", index=False)

display(paper_overall)
display(paper_per_srs)


## 4) Export LaTeX tables


In [ ]:
latex_overall = paper_overall.to_latex(index=False, float_format=lambda x: f"{x:.4f}")
latex_per_srs = paper_per_srs.to_latex(index=False, float_format=lambda x: f"{x:.4f}")

(RESULTS_TABLES / "paper_overall_results.tex").write_text(latex_overall, encoding="utf-8")
(RESULTS_TABLES / "paper_per_srs_results.tex").write_text(latex_per_srs, encoding="utf-8")

print(latex_overall)
print("Saved .tex files in", RESULTS_TABLES)


## 5) Plot figures for paper


In [ ]:
plt.figure(figsize=(8, 4))
vals = paper_overall.set_index("Metric")["Value"]
vals.plot(kind="bar")
plt.title("Overall Retrieval/Generation/E2E Metrics")
plt.ylabel("Score")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
fig1 = RESULTS_FIGS / "paper_overall_metrics.png"
plt.savefig(fig1, dpi=200)
plt.show()

plt.figure(figsize=(6, 4))
failure_breakdown.set_index("failure_type")["count"].plot(kind="bar")
plt.title("Failure Breakdown")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
fig2 = RESULTS_FIGS / "paper_failure_breakdown.png"
plt.savefig(fig2, dpi=200)
plt.show()

display(metrics_by_category)
print("Saved figures:")
print("-", fig1)
print("-", fig2)
